# RabTech Task 5 — Deep Learning / NLP Text Classifier

## TensorFlow / Keras Sentiment Classification with IMDB Reviews

This notebook implements the Task 5 brief as a complete NLP pipeline:

- Tokenize and vectorize text with **TensorFlow Keras `TextVectorization`**
- Build a multi-layer neural network with **Dropout** and **Batch Normalization**
- Train with **EarlyStopping** and **ReduceLROnPlateau** callbacks
- Validate and evaluate the model on unseen test data
- Plot **training/validation loss and accuracy convergence curves**
- Test the trained model on **unseen custom text inputs**
- Report **classification confidence**
- Save the trained model, vocabulary, metrics, plots, and sample inference outputs

Dataset: **IMDB movie reviews** (binary sentiment classification).

> The Task 5 brief allows either a Deep Learning neural-network pipeline or an NLP pipeline. A genuine text-classification dataset is used here so every NLP requirement is directly visible in the notebook.


In [ ]:
# Run this cell first.
%pip install -q tensorflow pandas numpy matplotlib scikit-learn


In [ ]:
from pathlib import Path
import json
import random
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)

print("TensorFlow version:", tf.__version__)
print("Random seed:", SEED)
print("Artifacts directory:", ARTIFACT_DIR.resolve())


## 1. Load and inspect the text dataset

The IMDB dataset contains movie reviews with binary sentiment labels:
- `0` = negative
- `1` = positive

The TensorFlow dataset loader provides integer-tokenized reviews. We decode those sequences back to normalized text so that the notebook explicitly demonstrates the requested **tokenization/vectorization stage** with Keras `TextVectorization`.


In [ ]:
# Load IMDB integer-encoded reviews.
NUM_WORDS = 20000
(train_seq, y_train_full), (test_seq, y_test) = tf.keras.datasets.imdb.load_data(
    num_words=NUM_WORDS,
    seed=SEED
)

word_index = tf.keras.datasets.imdb.get_word_index()
reverse_word_index = {idx + 3: word for word, idx in word_index.items()}
reverse_word_index[0] = "<PAD>"
reverse_word_index[1] = "<START>"
reverse_word_index[2] = "<UNK>"
reverse_word_index[3] = "<UNUSED>"

def decode_review(sequence):
    return " ".join(reverse_word_index.get(int(i), "<UNK>") for i in sequence)

train_text_full = np.array([decode_review(seq) for seq in train_seq], dtype=object)
test_text = np.array([decode_review(seq) for seq in test_seq], dtype=object)

print("Training reviews:", len(train_text_full))
print("Test reviews:", len(test_text))
print("Example label:", int(y_train_full[0]))
print("Example decoded review:")
print(train_text_full[0][:700])


In [ ]:
# Stratified train/validation split.
X_train_text, X_val_text, y_train, y_val = train_test_split(
    train_text_full,
    y_train_full,
    test_size=0.20,
    stratify=y_train_full,
    random_state=SEED
)

print("Train:", len(X_train_text))
print("Validation:", len(X_val_text))
print("Test:", len(test_text))
print("Train positive rate:", round(float(np.mean(y_train)), 4))
print("Validation positive rate:", round(float(np.mean(y_val)), 4))
print("Test positive rate:", round(float(np.mean(y_test)), 4))


## 2. Tokenize and vectorize the text

`TextVectorization` learns its vocabulary **only from the training split**. This keeps validation/test text from influencing the learned vocabulary.

The layer performs:
1. text standardization
2. tokenization
3. integer vectorization
4. fixed-length sequence padding/truncation


In [ ]:
MAX_TOKENS = 20000
SEQUENCE_LENGTH = 250

vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=MAX_TOKENS,
    output_mode="int",
    output_sequence_length=SEQUENCE_LENGTH,
    standardize="lower_and_strip_punctuation"
)

vectorizer.adapt(tf.data.Dataset.from_tensor_slices(X_train_text).batch(256))

vocabulary = vectorizer.get_vocabulary()
print("Vocabulary size:", len(vocabulary))
print("First 20 vocabulary tokens:", vocabulary[:20])

X_train_vec = vectorizer(tf.constant(X_train_text))
X_val_vec = vectorizer(tf.constant(X_val_text))
X_test_vec = vectorizer(tf.constant(test_text))

print("Vectorized train shape:", X_train_vec.shape)
print("Vectorized validation shape:", X_val_vec.shape)
print("Vectorized test shape:", X_test_vec.shape)

# Save vocabulary as a reproducibility artifact.
pd.DataFrame({"token": vocabulary}).to_csv(
    ARTIFACT_DIR / "vocabulary.csv", index=False
)


## 3. Build the multi-layer neural network

Architecture:

`Input → Embedding → GlobalAveragePooling1D → Dense + BatchNorm + ReLU → Dropout → Dense + BatchNorm + ReLU → Dropout → Sigmoid output`

This directly covers the brief's requirement for a **multi-layer neural network with Batch Normalization and Dropout**.


In [ ]:
EMBEDDING_DIM = 128

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(SEQUENCE_LENGTH,), dtype=tf.int64, name="token_ids"),
    tf.keras.layers.Embedding(
        input_dim=len(vocabulary),
        output_dim=EMBEDDING_DIM,
        name="embedding"
    ),
    tf.keras.layers.GlobalAveragePooling1D(name="global_average_pooling"),

    tf.keras.layers.Dense(128, activation="relu", name="dense_1"),
    tf.keras.layers.BatchNormalization(name="batch_norm_1"),
    tf.keras.layers.Dropout(0.35, name="dropout_1"),

    tf.keras.layers.Dense(64, activation="relu", name="dense_2"),
    tf.keras.layers.BatchNormalization(name="batch_norm_2"),
    tf.keras.layers.Dropout(0.30, name="dropout_2"),

    tf.keras.layers.Dense(1, activation="sigmoid", name="sentiment_output")
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name="accuracy"),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
        tf.keras.metrics.AUC(name="auc")
    ]
)

model.summary()


## 4. Train with validation and early-stopping callbacks

Callbacks used:
- **EarlyStopping**: stops training when validation loss stops improving and restores the best weights.
- **ReduceLROnPlateau**: lowers the learning rate when validation loss plateaus.
- **CSVLogger**: stores the training history as an artifact.

The maximum epoch count is deliberately bounded; early stopping determines the effective training duration.


In [ ]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=1,
    min_lr=1e-6,
    verbose=1
)

csv_logger = tf.keras.callbacks.CSVLogger(
    ARTIFACT_DIR / "training_history.csv",
    append=False
)

history = model.fit(
    X_train_vec,
    np.asarray(y_train),
    validation_data=(X_val_vec, np.asarray(y_val)),
    epochs=15,
    batch_size=128,
    callbacks=[early_stopping, reduce_lr, csv_logger],
    verbose=1
)

print("Epochs actually completed:", len(history.history["loss"]))


## 5. Training and validation convergence curves

The Task 5 expected proof explicitly asks for training curves. The notebook produces:
- loss convergence
- accuracy convergence

The plots are saved under `artifacts/`.


In [ ]:
history_df = pd.DataFrame(history.history)
history_df.to_csv(ARTIFACT_DIR / "training_history_copy.csv", index=False)

# Loss curve
plt.figure(figsize=(8, 5))
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Binary Cross-Entropy Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / "loss_convergence.png", dpi=180, bbox_inches="tight")
plt.show()

# Accuracy curve
plt.figure(figsize=(8, 5))
plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training vs Validation Accuracy")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / "accuracy_convergence.png", dpi=180, bbox_inches="tight")
plt.show()


## 6. Evaluate on the unseen test set

The test set is kept separate from training and validation. We report:
- Accuracy
- Precision
- Recall
- F1-score
- ROC-AUC
- Confusion matrix
- Classification report


In [ ]:
test_eval = model.evaluate(
    X_test_vec,
    np.asarray(y_test),
    verbose=0,
    return_dict=True
)

test_prob = model.predict(X_test_vec, batch_size=256, verbose=0).ravel()
test_pred = (test_prob >= 0.5).astype(int)

metrics = {
    "accuracy": accuracy_score(y_test, test_pred),
    "precision": precision_score(y_test, test_pred, zero_division=0),
    "recall": recall_score(y_test, test_pred, zero_division=0),
    "f1": f1_score(y_test, test_pred, zero_division=0),
    "roc_auc": roc_auc_score(y_test, test_prob)
}

print("Test metrics:")
display(pd.DataFrame([metrics]).round(4))

print("\nClassification report:")
print(classification_report(
    y_test,
    test_pred,
    target_names=["negative", "positive"],
    digits=4,
    zero_division=0
))

cm = confusion_matrix(y_test, test_pred)
cm_df = pd.DataFrame(
    cm,
    index=["Actual Negative", "Actual Positive"],
    columns=["Predicted Negative", "Predicted Positive"]
)
display(cm_df)

# Save metrics and confusion matrix.
pd.DataFrame([metrics]).to_csv(
    ARTIFACT_DIR / "test_metrics.csv", index=False
)
cm_df.to_csv(ARTIFACT_DIR / "confusion_matrix.csv")


In [ ]:
# Confusion matrix plot.
plt.figure(figsize=(6, 5))
plt.imshow(cm)
plt.title("Confusion Matrix — IMDB Sentiment")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.xticks([0, 1], ["Negative", "Positive"])
plt.yticks([0, 1], ["Negative", "Positive"])

for i in range(2):
    for j in range(2):
        plt.text(j, i, int(cm[i, j]), ha="center", va="center")

plt.tight_layout()
plt.savefig(ARTIFACT_DIR / "confusion_matrix.png", dpi=180, bbox_inches="tight")
plt.show()


## 7. Unseen sample inference with classification confidence

The following reviews are **not part of the IMDB training/test dataset**. The model predicts a sentiment probability and converts it into a class with a confidence score.


In [ ]:
unseen_samples = [
    "An emotional, beautifully acted movie with a memorable story. I loved every minute of it.",
    "The plot was boring, the acting was weak, and I regretted watching this film.",
    "A surprisingly enjoyable film with great performances and a satisfying ending.",
    "The movie was confusing and painfully slow, with characters that were impossible to care about."
]

unseen_vec = vectorizer(tf.constant(unseen_samples))
unseen_prob = model.predict(unseen_vec, verbose=0).ravel()

inference_rows = []
for text, prob in zip(unseen_samples, unseen_prob):
    label = "positive" if prob >= 0.5 else "negative"
    confidence = float(prob if prob >= 0.5 else 1.0 - prob)
    inference_rows.append({
        "text": text,
        "predicted_class": label,
        "positive_probability": float(prob),
        "confidence": confidence
    })

inference_df = pd.DataFrame(inference_rows)
display(inference_df.round(4))

inference_df.to_csv(
    ARTIFACT_DIR / "sample_inference_outputs.csv", index=False
)


## 8. Save the complete trained model

The saved Keras model contains the neural-network weights and architecture. The vocabulary is saved separately as `vocabulary.csv` for reproducibility.


In [ ]:
model_path = ARTIFACT_DIR / "imdb_sentiment_model.keras"
model.save(model_path)

run_metadata = {
    "task": "RabTech Task 5",
    "framework": "TensorFlow / Keras",
    "dataset": "IMDB movie reviews",
    "task_type": "binary text classification",
    "random_seed": SEED,
    "max_tokens": MAX_TOKENS,
    "sequence_length": SEQUENCE_LENGTH,
    "embedding_dimension": EMBEDDING_DIM,
    "epochs_requested": 15,
    "epochs_completed": len(history.history["loss"]),
    "test_metrics": {k: float(v) for k, v in metrics.items()},
    "model_path": str(model_path),
    "vocabulary_path": str(ARTIFACT_DIR / "vocabulary.csv")
}

with open(ARTIFACT_DIR / "run_metadata.json", "w", encoding="utf-8") as f:
    json.dump(run_metadata, f, indent=2)

print("Saved model:", model_path)
print("Saved metadata:", ARTIFACT_DIR / "run_metadata.json")


## 9. Final Task 5 checklist

| Requirement from Task 5 brief | Evidence |
|---|---|
| Tokenize and vectorize text | Keras `TextVectorization` |
| Deep-learning neural network | Multi-layer Keras Sequential model |
| Dropout | Two Dropout layers |
| Batch Normalization | Two BatchNormalization layers |
| Early stopping callback | `EarlyStopping(restore_best_weights=True)` |
| Validation | Stratified train/validation split |
| Training curves | Loss and accuracy convergence plots |
| Unseen sample testing | Four custom reviews |
| Classification confidence | Positive probability + confidence |
| Expected proof | Jupyter notebook + saved curves + inference table |
| Reproducibility | Fixed seed, requirements, metadata, saved vocabulary/model |

### Submission-ready artifacts

After **Run All** completes successfully, the `artifacts/` folder should contain:

- `imdb_sentiment_model.keras`
- `vocabulary.csv`
- `training_history.csv`
- `training_history_copy.csv`
- `loss_convergence.png`
- `accuracy_convergence.png`
- `test_metrics.csv`
- `confusion_matrix.csv`
- `confusion_matrix.png`
- `sample_inference_outputs.csv`
- `run_metadata.json`

**Important:** Do not submit the notebook until every cell has been run successfully and the final inference table is visible.
